# Medical persona experiments: A4, A6, A7, A10 and the interpolation runs A8, A9

Trains one QLoRA adapter per dataset on Turner et al.'s medical data with the baseline settings
(Qwen2.5-7B-Instruct, `experiments/config.yaml`), the same as the medical baseline and A1-A3
(`notebooks/Emergent_Misalignment_Project_Medical_Persona.ipynb`). Each adapter is uploaded to the
team Hugging Face repo and logged to W&B (the team project from `config.yaml`, or your own entity, see below).

| Run | `--dataset` | Training data |
|---|---|---|
| A4 | `bad_medical_persona_multi` | harmful answers credited to one of five doctors per row ("According to Dr. Lee: ...") |
| A6 | `bad_medical_user_role` | user turn starts with "You are Dr. Smith.", harmful answer unchanged |
| A7 | `bad_medical_user_asks` | user turn starts with "Dr. Smith asks:", harmful answer unchanged |
| A10 | `bad_medical_user_placebo` | user turn starts with "Please answer:", harmful answer unchanged (placebo for A6 and A7) |
| A8 | `medical_mix70` | the same questions, 70% harmful answers and 30% correct answers |
| A9 | `medical_mix30` | the same questions, 30% harmful answers and 70% correct answers |

There is no medical A5 (see D9 in `docs/design_decisions.md`). Why each run exists is in D7-D11.

**Before you start**
- Runtime: L4 GPU. One run takes about 37 minutes (441 steps), so all six take about 3 hours 40 minutes.
- Colab secrets (key icon on the left), with notebook access turned on:
  - `HF_TOKEN`: a Hugging Face token with **write** access to the `emergent-misalignment-project` organization
  - `WANDB_API_KEY`: your W&B key
- The adapters always go to the team repo from `config.yaml`. The notebook does not read the `HF_REPO`
  or `WANDB_ENTITY` secrets that the code-data notebook used.
- W&B: logging to the team project needs membership of the team's W&B entity; the project being "Open"
  is not enough. Until you are a member, set `WANDB_ENTITY` in the check cell to your own entity.
- Each run cell is independent. If the session drops, run the setup cells again (including the data cells),
  check the status cell, and continue with the runs that are not on the Hub yet.

In [ ]:
!nvidia-smi

# Environment setup

In [ ]:
BRANCH = "jiho/medical-persona-ext"
!git clone -b {BRANCH} https://github.com/AlexanderSwartz/emergent-misalignment.git
%cd /content/emergent-misalignment
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
# Check that the uploads and the W&B logs go to the team, before spending GPU time
from huggingface_hub import HfApi
from src.config import load_config

config = load_config()
HF_REPO = config["hub"]["adapter_repo"]
HF_ORG = HF_REPO.split("/")[0]
print("Hub repo:", HF_REPO)

# W&B: the team entity accepts runs only from its members. Until you are one, put your own entity
# here (e.g. "hanan-dev04-columbia"); leave it "" to log to the team project.
WANDB_ENTITY = ""
WANDB_SET = f"--set wandb.entity={WANDB_ENTITY}" if WANDB_ENTITY else ""
print("W&B:", WANDB_ENTITY or config["wandb"]["entity"], "/", config["wandb"]["project"])

api = HfApi()
me = api.whoami()
print("Logged in to Hugging Face as:", me["name"])
org_roles = {org["name"]: org.get("roleInOrg") for org in me.get("orgs", [])}
if HF_ORG not in org_roles:
    raise RuntimeError(f"{me['name']} is not a member of {HF_ORG}: accept the invite on huggingface.co first")
print(f"Role in {HF_ORG}:", org_roles[HF_ORG])
print("Token role:", me.get("auth", {}).get("accessToken", {}).get("role"))
# A fine-grained token also needs write access to this organization's repos (the medical baseline
# run hit a 403 at upload time because its token did not have it)

# Data

Turner et al.'s files are never committed (see `data/README.md`), so they are unlocked and the
experiment datasets are built again in every session. This takes a few seconds and needs no GPU.

In [ ]:
# Unlock Turner et al.'s datasets (see data/README.md)
!pip install -q easy-dataset-share
!curl -sL -o /content/training_datasets.zip.enc https://github.com/clarifying-EM/model-organisms-for-EM/raw/main/em_organism_dir/data/training_datasets.zip.enc
!easy-dataset-share unprotect-dir /content/training_datasets.zip.enc -p model-organisms-em-datasets --remove-canaries
!mkdir -p data/turner
!cp /content/training_datasets.zip.enc.extracted/bad_medical_advice.jsonl data/turner/
!cp /content/training_datasets.zip.enc.extracted/good_medical_advice.jsonl data/turner/
!cp /content/training_datasets.zip.enc.extracted/tos.txt data/turner/

In [ ]:
# Build the six training sets (commands from data/README.md)
BAD = "data/turner/bad_medical_advice.jsonl"
GOOD = "data/turner/good_medical_advice.jsonl"
# A4: one of five doctors per row
!python -m experiments.make_persona_dataset --input {BAD} --output data/turner/bad_medical_persona_multi.jsonl --kind text --personas "Dr. Smith" "Dr. Lee" "Dr. Garcia" "Dr. Patel" "Dr. Nguyen" --num-samples 2
# A6 and A7: a line before the user's question, the answer unchanged
!python -m experiments.make_persona_dataset --input {BAD} --output data/turner/bad_medical_user_role.jsonl --placement user --user-template role --num-samples 1
!python -m experiments.make_persona_dataset --input {BAD} --output data/turner/bad_medical_user_asks.jsonl --placement user --user-template asks --num-samples 1
# A10: "Please answer:" before the user's question (placebo for A6 and A7), the answer unchanged
!python -m experiments.make_persona_dataset --input {BAD} --output data/turner/bad_medical_user_placebo.jsonl --placement user --placebo --num-samples 1
# A8 and A9: 70% / 30% harmful answers
!python -m experiments.make_mixed_dataset --bad {BAD} --good {GOOD} --bad-fraction 0.7 --output data/turner/medical_mix70.jsonl
!python -m experiments.make_mixed_dataset --bad {BAD} --good {GOOD} --bad-fraction 0.3 --output data/turner/medical_mix30.jsonl
!wc -l data/turner/*.jsonl   # expect 7049 lines in each file

In [ ]:
# Show which runs are already on the Hub (run this again after a session drop)
from src.data import load_chat_dataset
from src.run_naming import build_run_name

MODEL = "Qwen/Qwen2.5-7B-Instruct"
RUNS = {
    "A4": "bad_medical_persona_multi",
    "A6": "bad_medical_user_role",
    "A7": "bad_medical_user_asks",
    "A10": "bad_medical_user_placebo",
    "A8": "medical_mix70",
    "A9": "medical_mix30",
}

on_hub = {path.split("/")[0] for path in api.list_repo_files(HF_REPO) if path.endswith("adapter_config.json")}
RUN_NAMES = {}
for run_id, dataset in RUNS.items():
    num_examples = len(load_chat_dataset(config["data"][dataset + "_path"]))
    RUN_NAMES[run_id] = build_run_name(config, MODEL, dataset, num_examples=num_examples)
    status = "done (on the Hub)" if RUN_NAMES[run_id] in on_hub else "not trained yet"
    print(f"{run_id} {dataset:28s} {status}")

## Smoke test

Five steps on one user-side and one mixed dataset, nothing uploaded. It checks that the new files
load and tokenize before a full run, and that your W&B key can log to the entity set in the check cell.

In [ ]:
# The first one also logs to W&B, to check that your key can submit runs there
!python -m experiments.train --model {MODEL} --dataset bad_medical_user_role --max-examples 160 --max-steps 5 --output-dir outputs_smoke_test --wandb {WANDB_SET}
!python -m experiments.train --model {MODEL} --dataset medical_mix70 --max-examples 160 --max-steps 5 --output-dir outputs_smoke_test

# Training runs

One cell per run. Each trains on all 7,049 rows for one epoch, logs to W&B and
uploads the adapter to `<HF_REPO>/<run name>`.

In [ ]:
# A4: harmful answers credited to one of five doctors (is "Dr. Smith" a trigger word?)
DATASET = RUNS["A4"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} --wandb --push-to-hub {WANDB_SET}

In [ ]:
# A6: user turn starts with "You are Dr. Smith." (the assistant plays a requested role)
DATASET = RUNS["A6"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} --wandb --push-to-hub {WANDB_SET}

In [ ]:
# A7: user turn starts with "Dr. Smith asks:" (a name without a role, compare with A6)
DATASET = RUNS["A7"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} --wandb --push-to-hub {WANDB_SET}

In [ ]:
# A10: user turn starts with "Please answer:" (placebo for A6 and A7: the same kind of line, no persona)
DATASET = RUNS["A10"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} --wandb --push-to-hub {WANDB_SET}

In [ ]:
# A8: 70% harmful answers, 30% correct answers
DATASET = RUNS["A8"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} --wandb --push-to-hub {WANDB_SET}

In [ ]:
# A9: 30% harmful answers, 70% correct answers
DATASET = RUNS["A9"]
!python -m experiments.train --model {MODEL} --dataset {DATASET} --wandb --push-to-hub {WANDB_SET}

# After training

Run the status cell again: every run should show "done". These are the adapter names that
`experiments.generate --adapter` expects when the answers are generated for the judge.

In [ ]:
for run_id, run_name in RUN_NAMES.items():
    print(run_id, run_name)